# 01 · Calidad de datos
Objetivo: conocer volumen, rango de fechas y problemas antes de analizar. **Todo hallazgo de calidad se anota en `docs/hallazgos.md`.**

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import config, load, quality, flow, features, models

# Fuente de datos:
#   "db"     -> base de datos del reto (FortiClient VPN conectada + archivo .env)
#   "raw"    -> archivos exportados en data/raw
#   "sample" -> datos sintéticos de prueba (scripts/generar_datos_sinteticos.py)
SOURCE = "db"
DATA_DIR = ROOT / "data" / ("sample" if SOURCE == "sample" else "raw")

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
if SOURCE == "db":
    from src import db
    db.test_connection()
    t = db.load_all()   # copia local en data/interim; db.load_all(refresh=True) para volver a descargar
else:
    t = load.load_all(DATA_DIR)

In [ ]:
if SOURCE == 'db':
    display(db.row_counts())

## Resumen por tabla

In [ ]:
for name, df in t.items():
    print(f'\n=== {name} ===')
    display(quality.summary(df))

## Intervalos, integridad y duplicados

In [ ]:
rep = quality.full_report(t)
pd.DataFrame({k: v for k, v in rep.items() if 'n' in v}).T

In [ ]:
{k: v.get('fk', v) for k, v in rep.items() if k not in ('visit_occurrence',)}

## Conceptos presentes (para mapear con Athena / tabla CONCEPT)

In [ ]:
# Nombres de los conceptos desde la tabla CONCEPT de la BD (si existe)
if SOURCE == 'db':
    try:
        display(db.concept_map())
    except Exception as e:
        print('No se pudo leer CONCEPT:', e)

In [ ]:
for tab, col in [('visit_occurrence','visit_concept_id'), ('visit_occurrence','visit_type_concept_id'),
                 ('visit_detail','visit_detail_concept_id'), ('person','gender_concept_id')]:
    if tab in t:
        print(f'\n{tab}.{col}')
        display(t[tab][col].value_counts().head(20))

## Volumen en el tiempo

In [ ]:
vo = t['visit_occurrence']
vo.set_index('visit_start_datetime').resample('W').size().plot(figsize=(12,3), title='Visitas por semana');

## Conclusiones de calidad
- _(anotar aquí: % de fechas 00:00, visitas abiertas, huérfanos, decisiones de limpieza)_